# 🧬 Genomic Context Retrieval for Sequence Classification
Retrieves a transcript record (pinned to a specific version), locates it on the parent
chromosome, and fetches a genomic window with flanking regions for feature computation
(repetitive elements, non-B DNA motifs, etc.).


## 1. Setup

In [1]:
from Bio import Entrez, SeqIO
from Bio.SeqFeature import CompoundLocation
import pandas as pd

Entrez.email = 'your.email@example.com'

# Pin to the exact version used in your dataset — do NOT update to .2
TRANSCRIPT_ACC  = 'XM_008105585.1'
GENOMIC_ACC     = 'NC_014776.1'   # parent chromosome, from COMMENT field
FLANK_BP        = 1000            # flanking window size (bp) on each side


## 2. Fetch the Transcript Record (pinned version)
We fetch `.1` explicitly. Biopython will return this exact version even though `.2` exists.

In [2]:
handle = Entrez.efetch(db='nucleotide', id=TRANSCRIPT_ACC, rettype='gb', retmode='text')
tx_record = SeqIO.read(handle, 'genbank')
handle.close()

print(f'Transcript  : {tx_record.id}')
print(f'Description : {tx_record.description}')
print(f'Length      : {len(tx_record.seq)} bp')
print(f'Organism    : {tx_record.annotations.get("organism")}')


Transcript  : XM_008105585.1
Description : PREDICTED: Anolis carolinensis chromosome 1 open reading frame, human C14orf2 (c1h14orf2), mRNA
Length      : 443 bp
Organism    : Anolis carolinensis


## 3. Extract Transcript Metadata
Parse the gene name, Gene ID, protein product, and CDS coordinates from the transcript record.

In [3]:
info = {'transcript_acc': tx_record.id, 'length': len(tx_record.seq)}

for feature in tx_record.features:
    if feature.type == 'gene':
        info['gene_name'] = feature.qualifiers.get('gene', ['?'])[0]
        for xref in feature.qualifiers.get('db_xref', []):
            if xref.startswith('GeneID:'):
                info['gene_id'] = xref.split(':')[1]

    if feature.type == 'CDS':
        info['protein_id'] = feature.qualifiers.get('protein_id', ['?'])[0]
        info['product']    = feature.qualifiers.get('product',    ['?'])[0]
        # CDS coordinates on the transcript (0-based half-open)
        info['cds_tx_start'] = int(feature.location.start) + 1  # 1-based
        info['cds_tx_end']   = int(feature.location.end)

for k, v in info.items():
    print(f'  {k:<20} : {v}')


  transcript_acc       : XM_008105585.1
  length               : 443
  gene_name            : c1h14orf2
  gene_id              : 103278012
  protein_id           : XP_008103792.1
  product              : 6.8 kDa mitochondrial proteolipid
  cds_tx_start         : 120
  cds_tx_end           : 290


## 4. Find Genomic Coordinates of the Transcript
We search the parent chromosome record (`NC_014776.1`) for the `mRNA` feature
matching our transcript accession. This gives us chromosome-relative coordinates
and reveals the exon structure via compound locations.

> **Note:** `NC_014776.1` is a large chromosome. We use `Entrez.esearch` +
> `elink` to locate the transcript's position without downloading the full chromosome.

In [8]:
# Strategy 1: search without version number
handle = Entrez.esearch(db='nucleotide', term='XM_008105585[accn]')
res    = Entrez.read(handle)
handle.close()
print(res['IdList'])  # may return the .2 UID — that's fine, we only need it for elink
tx_uid = res['IdList'][0]

['2672791429']


## 5. Parse Exon Boundaries from the Genomic Record
Fetch only the annotation (no sequence yet) for a region of the chromosome.
We use `rettype='gb'` with `seq_start`/`seq_stop` around the locus.

Since we don't yet know the exact position, we first fetch the full feature
table of `NC_014776.1` filtered by our transcript ID using `efetch` with `rettype=ft`.

In [10]:
# Fetch the feature table for the chromosome — lightweight, no sequence
handle = Entrez.efetch(db='nucleotide', id=GENOMIC_ACC,
                       rettype='ft', retmode='text')
ft_text = handle.read()
handle.close()

# Parse lines mentioning our transcript
lines = ft_text.split('\n')
relevant, capture = [], False
for line in lines:
    if TRANSCRIPT_ACC.split('.')[0] in line:  # match on base accession
        capture = True
    if capture:
        relevant.append(line)
        if line.strip() == '' and len(relevant) > 2:
            break

print('\n'.join(relevant[:40]))


			transcript_id	ref|XM_008105585.1|
			gene	c1h14orf2
			db_xref	GeneID:103278012
5894276	5894399	CDS
5895218	5895244
5897455	5897474
			product	6.8 kDa mitochondrial proteolipid
			protein_id	ref|XP_008103792.1|
			gene	c1h14orf2
			db_xref	GeneID:103278012
5924714	5999932	gene
			gene	ppp1r13b
			db_xref	GeneID:100559628
5924714	5925009	mRNA
5946685	5946832
5952102	5952221
5956549	5956613
5978967	5979068
5980601	5980775
5982520	5982716
5982918	5983082
5983499	5983679
5984502	5984654
5990335	5990504
5990721	5991222
5993012	>5993642
5994820	5994957
5995048	5995181
5995330	5995496
5995938	5996137
5999301	5999932
			product	protein phosphatase 1 regulatory subunit 13B
			transcript_id	ref|XM_016990677.1|
			gene	ppp1r13b
			note	The sequence of the model RefSeq transcript was modified relative to this genomic sequence to represent the inferred CDS: added 136 bases not found in genome assembly
			exception	annotated by transcript or proteomic data
			inference	similar to RNA sequence, mR

## 6. Extract Genomic Locus Coordinates
Parse the mRNA feature coordinates for our transcript from the full chromosome GenBank record.
We fetch only the annotation around a known region using `rettype='gb'` without sequence (`rettype='gbwithparts'` is avoided for large chromosomes).

In [11]:
import re

# Extract coordinates from feature table text
# Feature table format: coordinates are in the first column
locus_start, locus_end, strand, exon_coords = None, None, '+', []

i = 0
while i < len(lines):
    line = lines[i]
    # Detect mRNA feature block for our transcript
    if '\tmRNA\t' in line or (line.strip().startswith('mRNA') and i > 0):
        # Look ahead for transcript_id qualifier
        block = '\n'.join(lines[i:i+30])
        if TRANSCRIPT_ACC.split('.')[0] in block:
            # Parse coordinates from this block
            coord_matches = re.findall(r'(complement\()?join\(([\d\.\.,]+)\)?', block)
            # Also handle simple coordinates
            simple = re.findall(r'(complement\()?(\d+)\.\.(\d+)', line)
            for m in simple:
                if m[0]: strand = '-'
                exon_coords.append((int(m[1]), int(m[2])))
    i += 1

# Fallback: parse all exon/mRNA coordinates from ft_text for our transcript
# More robust approach using regex on the full feature table
tx_base = TRANSCRIPT_ACC.rsplit('.', 1)[0]  # 'XM_008105585'
blocks  = re.split(r'(?=\n\d|\n<\d|\n>\d)', ft_text)
for block in blocks:
    if tx_base in block and 'mRNA' in block:
        coord_lines = re.findall(r'[<>]?(\d+)\.\.?[<>]?(\d+)', block)
        if coord_lines:
            all_coords  = [(int(a), int(b)) for a, b in coord_lines]
            locus_start = min(c[0] for c in all_coords)
            locus_end   = max(c[1] for c in all_coords)
            exon_coords = all_coords
            strand      = '-' if 'complement' in block else '+'
            print(f'Locus on {GENOMIC_ACC}: {locus_start}..{locus_end} ({strand})')
            print(f'Exon coordinates ({len(exon_coords)} exons):')
            for j, (s, e) in enumerate(exon_coords, 1):
                print(f'  Exon {j}: {s}..{e}  ({e - s + 1} bp)')
            break


## 7. Build the Exon Table (GTF-like, genomic coordinates)
Now we have chromosome-relative, 1-based exon coordinates consistent with the `.1` annotation.

In [12]:
rows = []
for i, (s, e) in enumerate(exon_coords, 1):
    rows.append({
        'seqname'      : GENOMIC_ACC,
        'source'       : 'GenBank_v1',
        'feature'      : 'exon',
        'start'        : s,
        'end'          : e,
        'score'        : '.',
        'strand'       : strand,
        'frame'        : '.',
        'gene_id'      : info.get('gene_name', '?'),
        'transcript_id': TRANSCRIPT_ACC,
        'exon_number'  : i
    })

df_exons = pd.DataFrame(rows)
df_exons['length'] = df_exons['end'] - df_exons['start'] + 1
print(df_exons.to_string(index=False))
df_exons.to_csv('exons_genomic_coords.csv', index=False)


KeyError: 'end'

## 8. Fetch the Genomic Window with Flanks
We now fetch the genomic sequence spanning the full locus ± `FLANK_BP` bases.
This window will be used for computing repetitive elements and non-B DNA motifs.

> ⚠️ `seq_start`/`seq_stop` in `efetch` are **1-based inclusive** coordinates.

In [ ]:
window_start = max(1, locus_start - FLANK_BP)
window_end   = locus_end + FLANK_BP

print(f'Fetching genomic window: {GENOMIC_ACC}:{window_start}-{window_end}')
print(f'Window size: {window_end - window_start + 1} bp')

handle = Entrez.efetch(
    db       = 'nucleotide',
    id       = GENOMIC_ACC,
    rettype  = 'fasta',
    retmode  = 'text',
    seq_start = window_start,
    seq_stop  = window_end
)
genomic_fasta = handle.read()
handle.close()

print(genomic_fasta[:300])  # preview


## 9. Save the Genomic Window
Save the sequence and record the coordinate offset so that all downstream feature
positions can be re-anchored back to chromosome coordinates.

In [ ]:
# Save FASTA
fasta_file = f'genomic_window_{TRANSCRIPT_ACC}.fa'
with open(fasta_file, 'w') as f:
    f.write(genomic_fasta)
print(f'Saved: {fasta_file}')

# Save coordinate metadata — critical for re-anchoring feature positions
import json
meta = {
    'transcript_acc'  : TRANSCRIPT_ACC,
    'genomic_acc'     : GENOMIC_ACC,
    'locus_start'     : locus_start,
    'locus_end'       : locus_end,
    'strand'          : strand,
    'window_start'    : window_start,   # offset: pos_in_fasta = chrom_pos - window_start + 1
    'window_end'      : window_end,
    'flank_bp'        : FLANK_BP,
    'exons_genomic'   : exon_coords,    # list of (start, end) in chrom coords
    'gene_name'       : info.get('gene_name'),
    'gene_id'         : info.get('gene_id'),
    'protein_id'      : info.get('protein_id'),
    'cds_tx_start'    : info.get('cds_tx_start'),
    'cds_tx_end'      : info.get('cds_tx_end'),
}
meta_file = f'genomic_window_{TRANSCRIPT_ACC}.meta.json'
with open(meta_file, 'w') as f:
    json.dump(meta, f, indent=2)
print(f'Saved: {meta_file}')
print('\nCoordinate offset formula:')
print(f'  position_in_fasta = chrom_position - {window_start} + 1')
print(f'  chrom_position    = position_in_fasta + {window_start} - 1')


## 10. Re-anchor Exon Coordinates to the FASTA Window
Convert chromosome-relative exon positions to window-relative positions.
These are the coordinates you will use when slicing features from the fetched sequence.

In [ ]:
df_exons['window_start'] = df_exons['start'] - window_start  # 0-based in window
df_exons['window_end']   = df_exons['end']   - window_start  # 0-based exclusive end

print('Exon positions within the fetched FASTA window (0-based):')
print(df_exons[['exon_number','start','end','window_start','window_end','length','strand']].to_string(index=False))

# Also save the transcript sequence itself from the fetched window
from Bio import SeqIO
import io
window_record = SeqIO.read(io.StringIO(genomic_fasta), 'fasta')
window_seq    = str(window_record.seq)

# Reconstruct transcript sequence from exons (forward strand example)
if strand == '+':
    tx_seq_reconstructed = ''.join(
        window_seq[row.window_start : row.window_end]
        for _, row in df_exons.iterrows()
    )
else:
    from Bio.Seq import Seq
    tx_seq_reconstructed = str(Seq(''.join(
        window_seq[row.window_start : row.window_end]
        for _, row in df_exons.sort_values('start', ascending=False).iterrows()
    )).reverse_complement())

print(f'\nReconstructed transcript length : {len(tx_seq_reconstructed)} bp')
print(f'Original transcript length      : {len(tx_record.seq)} bp')
print(f'Sequences match                 : {tx_seq_reconstructed.upper() == str(tx_record.seq).upper()}')


## Next Steps
With the genomic window saved and coordinates anchored, you are ready to:
- Run **RepeatMasker** on `genomic_window_XM_008105585.1.fa` to annotate repetitive elements
- Run **non-B DB** or **QGRS Mapper** for non-B DNA motifs (G4, Z-DNA, cruciform, mirror repeats)
- Use `window_start` offset from the `.meta.json` to convert any tool output back to chromosome coordinates
- Slice feature vectors relative to each exon's `window_start`/`window_end` for the classifier
